## נתונים מול מודל: הגרף המלא, savetxt

מגיע הזמן לאסוף הכל: נתונים עם שגיאות (`errorbar`), קו ההתאמה, שאריות, וסיכום מספרי (`chi^2_nu`, `R^2`) -- בגרף אחד, ולשמור את טבלת התוצאות לקובץ עם `np.savetxt`, בדיוק כמו שהתחלנו בשבוע 7.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

g = 9.8
df = pd.read_csv("lab_measurements.csv")
df_clean = df.dropna()
angles = sorted(df_clean["angle_deg"].unique())

x = np.array([np.sin(2*np.radians(a)) for a in angles])
y = np.array([df_clean[df_clean["angle_deg"] == a]["range_measured"].mean() for a in angles])
sigma_y = np.array([
    df_clean[df_clean["angle_deg"] == a]["range_measured"].std(ddof=1) / np.sqrt(len(df_clean[df_clean["angle_deg"] == a]))
    for a in angles
])

def linear_fit(x, y):
    x_bar, y_bar = x.mean(), y.mean()
    m = np.sum((x - x_bar) * (y - y_bar)) / np.sum((x - x_bar)**2)
    b = y_bar - m * x_bar
    return m, b

m, b = linear_fit(x, y)
x_line = np.linspace(0, 1, 100)
y_line = m*x_line + b
resid = y - (m*x + b)

chi2 = np.sum((resid/sigma_y)**2)
reduced_chi2 = chi2 / (len(x) - 2)
r2 = 1 - np.sum(resid**2) / np.sum((y - y.mean())**2)

### הגרף המלא: נתונים + התאמה + שאריות

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 6), height_ratios=[2, 1], sharex=True)

ax1.errorbar(x, y, yerr=sigma_y, fmt="o", label="מדידה (ממוצע לזווית)", capsize=3)
ax1.plot(x_line, y_line, "-", label=f"התאמה: R = {m:.2f}·sin(2θ) + {b:.2f}")
ax1.set_ylabel("Mean range [m]")
ax1.set_title(f"chi²_ν = {reduced_chi2:.2f}   R² = {r2:.3f}")
ax1.legend()

ax2.axhline(0, color="gray", linewidth=1)
ax2.errorbar(x, resid, yerr=sigma_y, fmt="o", color="tab:red")
ax2.set_xlabel("sin(2θ)")
ax2.set_ylabel("Residual [m]")

plt.tight_layout()
plt.show()

### שמירת טבלת התוצאות עם savetxt

בשבוע 7 שמרנו מערך פשוט. עכשיו יש לנו כמה עמודות -- `np.column_stack` בונה מהן טבלה דו-ממדית אחת, ו-`header` (עם `#` בתחילת השורה, אוטומטית ב-`savetxt`) מתעד מה כל עמודה.

In [ ]:
results_table = np.column_stack([angles, x, y, sigma_y, m*x + b, resid])
header = "angle_deg sin_2theta range_mean sigma_range fit_pred residual"

np.savetxt("fit_results.txt", results_table, header=header, fmt="%.4f")
print(open("fit_results.txt").read())

### באג נפוץ: header לא תואם לסדר העמודות בפועל

`savetxt` **לא בודק** שמספר המילים ב-`header` תואם למספר העמודות בפועל, ובטח שלא שהסדר נכון. אם משנים את סדר העמודות ב-`column_stack` בלי לעדכן את ה-`header` (או להפך), הקובץ נשמר בהצלחה, בלי שום שגיאה -- אבל מי שיקרא אותו מאוחר יותר יפרש עמודה כאילו היא עמודה אחרת לגמרי.

In [ ]:
results_table_reordered = np.column_stack([x, angles, y, sigma_y, m*x + b, resid])   # הוחלף סדר עמודה 1 ו-2!
header_unchanged = "angle_deg sin_2theta range_mean sigma_range fit_pred residual"          # לא עודכן בהתאם

np.savetxt("fit_results_BUGGY.txt", results_table_reordered, header=header_unchanged, fmt="%.4f")
print(open("fit_results_BUGGY.txt").read())
print("שימו לב: לפי הכותרת, העמודה הראשונה היא 'angle_deg' - אבל בפועל היא sin(2θ). אין שום שגיאה שמתריעה על כך.")

### נסו בעצמכם

טענו מחדש את `fit_results.txt` עם `np.loadtxt`, וודאו שהעמודה השנייה (אינדקס 1, `sin_2theta`) תואמת בדיוק ל-`x` המקורי.

In [ ]:
# reloaded = np.loadtxt("fit_results.txt")
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
reloaded = np.loadtxt("fit_results.txt")
print(np.allclose(reloaded[:, 1], x))
```
`````

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "למה חוסר-התאמה בין header לעמודות ב-savetxt הוא באג 'מסוכן' במיוחד?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "כי savetxt זורק שגיאה ומונע את השמירה", "correct": False, "feedback": "לא - savetxt לא בודק את התאמת ה-header כלל."},
            {"answer": "זה לא באג אמיתי, רק עניין קוסמטי", "correct": False, "feedback": "לא - זה עלול להוביל לניתוח שגוי לחלוטין של נתונים בעתיד."},
            {"answer": "כי הקובץ נשמר בהצלחה, נראה תקין, וניתן לטעינה - הטעות מתגלה רק כשמפרשים עמודה לפי שם שגוי", "correct": True, "feedback": "נכון."},
            {"answer": "כי savetxt מוחק אוטומטית עמודות עם header שגוי", "correct": False, "feedback": "לא — savetxt לא בודק ולא מוחק כלום; הוא פשוט כותב את המספרים ואת שורת הכותרת כטקסט, בלי לוודא התאמה ביניהם."}
        ]
    }
]
display_quiz(questions)

### תרגול עצמי

הוסיפו לטבלה עמודה נוספת: השארית המנורמלת (`resid/sigma_y`, "כמה סטיות תקן" כל נקודה רחוקה מהמודל), עדכנו את ה-`header` בהתאם, ושמרו מחדש. זהו בדיוק מדד הזיהוי הוויזואלי שהיה שימושי בסעיף 11.10 לזיהוי הנקודה החריגה.

In [ ]:
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
normalized_resid = resid / sigma_y
results_table_v2 = np.column_stack([angles, x, y, sigma_y, m*x + b, resid, normalized_resid])
header_v2 = "angle_deg sin_2theta range_mean sigma_range fit_pred residual normalized_residual"

np.savetxt("fit_results_v2.txt", results_table_v2, header=header_v2, fmt="%.4f")
print(open("fit_results_v2.txt").read())
```
`````